In [ ]:
num_classes = len(class_counts)

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:

# epochs = 10

# history = model.fit(
#     train_dataset,
#     validation_data = val_dataset,
#     epochs = epochs
# )

In [ ]:
#Create the callbacks

checkpoint_dir = "/home/user/Sync/backup/checkpoints"
backup_dir = "/home/user/Sync/backup/training_backup"

os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(backup_dir, exist_ok=True)

In [ ]:
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "best_model.keras"),
    monitor="val_loss",
    save_best_only=True,
    mode="min",
    verbose=1
)

backup_restore = tf.keras.callbacks.BackupAndRestore(
    backup_dir=backup_dir,
    save_freq="epoch"
)

#early stopping

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
    
)

In [ ]:


history = model.fit(
    train_dataset,
    validation_data = val_dataset,
    epochs = 20,
    callbacks = [model_checkpoint,
                 backup_restore,
                 early_stopping
                 ]
)

In [ ]:
print(os.listdir(checkpoint_dir))

In [ ]:
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['loss'], label = 'train loss')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.plot(history.history['val_loss'], label = 'val_loss')

plt.xlabel('Epoch')
plt.legend(loc='lower right')


In [ ]:
# evaluating on test
# test_loss, test_acc = model.evaluate(test_dataset, verbose=2)

In [ ]:
# prediction

# getting an image from test data

for images , labels in test_dataset.take(1):
    sample = images[0].numpy()
    true_label = label_encoder.classes_[tf.argmax(labels[0]).numpy()]


plt.imshow(sample)
plt.title(f"Actual class: {true_label}")
plt.axis("off")
plt.show()

#getting the best model from checkpoint

best_model = tf.keras.models.load_model(
    "/home/user/Sync/backup/checkpoints/best_model.keras"
)

#predicting
probas = best_model.predict( np.expand_dims(sample, axis=0), verbose=0 )[0]

predicted_index = np.argmax(probas)
predicted_class = label_encoder.classes_[predicted_index]
confidence = probas[predicted_index] * 100
print("Actual class:", true_label)
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.2f}%")

In [ ]:
!pip install  pydot
!apt-get -qq install graphviz

In [ ]:


plot_model(
    best_model,
    to_file="cnn_architecture.png",
    show_shapes=True,
    show_layer_names=True,
    expand_nested=True,
    dpi=100
)

# display(Image(filename="cnn_architecture.png"))

In [ ]:
# import shutil

# shutil.copy2(
#     "/kaggle/working/checkpoints/best_model.keras",
#     "/kaggle/working/best_model.keras"
# )